## Connect to the MySQL `supply_chain` database

In this step, we create a connection from Python to MySQL using SQLAlchemy.

**Why this matters:**
- It lets us run SQL queries directly from Python.
- The results are loaded into Pandas DataFrames for analytics and ML preparation.


In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# Your MySQL credentials
USER = "root"
PASSWORD = "YourPasswordHere"   # Put your password here
HOST = "localhost"
PORT = "3306"
DB = "supply_chain"

# Build connection string
connection_string = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB}"

# Create engine
engine = create_engine(connection_string)

# Quick smoke test
test_df = pd.read_sql("SELECT 1 AS ok;", engine)
print(test_df)


   ok
0   1


## Load the ML datasets from MySQL

In this step, we extract the three core datasets required for our machine learning workflow:
- the `Orders` table
- the `Order_Items` table
- the `Shipments` table

These tables contain transactional, item-level, and logistics information that will later be merged into a unified dataset.

In [5]:
orders = pd.read_sql_query("SELECT * FROM Orders;", engine)
order_items = pd.read_sql_query("SELECT * FROM Order_Items;", engine)
shipments = pd.read_sql_query("SELECT * FROM Shipments;", engine)

orders.head(), order_items.head(), shipments.head()


(   order_id  customer_id  warehouse_id  order_date      status  \
 0         1            1             1  2023-01-12   Delivered   
 1         2            5             1  2023-01-25     Shipped   
 2         3           12             2  2023-02-03  Processing   
 3         4            7             3  2023-02-18   Delivered   
 4         5            9             1  2023-03-02   Delivered   
 
                                       order_meta_xml          created_at  \
 0  <meta><priority>normal</priority><channel>onli... 2026-06-08 22:35:10   
 1  <meta><priority>normal</priority><channel>phon... 2026-06-08 22:35:10   
 2  <meta><priority>high</priority><channel>online... 2026-06-08 22:35:10   
 3  <meta><priority>normal</priority><channel>onli... 2026-06-08 22:35:10   
 4  <meta><priority>normal</priority><channel>emai... 2026-06-08 22:35:10   
 
            updated_at  
 0 2026-06-08 22:35:10  
 1 2026-06-08 22:35:10  
 2 2026-06-08 22:35:10  
 3 2026-06-08 22:35:10  
 4 2026

## ML Dataset 1  Classification: Predicting Order Cancellation

This dataset is designed for a classification problem where the goal is to predict 
whether an order will be cancelled. The dataset combines information from orders, 
customers, and order items, including:

- Customer type (from JSON)
- Order priority (from XML)
- Order channel (from XML)
- Total items and order value
- Number of products in the order
- Warehouse used

The target variable is the order status, which indicates whether the order was cancelled.

In [6]:
classification_df = pd.read_sql_query("""
SELECT
    o.order_id,
    o.customer_id,
    o.warehouse_id,
    o.status AS target_label,
    JSON_EXTRACT(c.customer_profile_json, '$.type') AS customer_type,
    EXTRACTVALUE(o.order_meta_xml, '/meta/priority') AS priority,
    EXTRACTVALUE(o.order_meta_xml, '/meta/channel') AS channel,
    SUM(oi.quantity) AS total_items,
    SUM(oi.quantity * oi.unit_price) AS order_value,
    COUNT(oi.product_id) AS num_products
FROM Orders o
JOIN Customers c ON o.customer_id = c.customer_id
JOIN Order_Items oi ON o.order_id = oi.order_id
GROUP BY 
    o.order_id, o.customer_id, o.warehouse_id, o.status,
    customer_type, priority, channel;
""", engine)

classification_df.head()


,order_id,customer_id,warehouse_id,target_label,customer_type,priority,channel,total_items,order_value,num_products
0,1,1,1,Delivered,"""retail""",normal,online,5.0,62.40,2
1,2,5,1,Shipped,"""retail""",normal,phone,5.0,604.35,2
2,3,12,2,Processing,"""business""",high,online,8.0,50.82,2
3,4,7,3,Delivered,"""retail""",normal,online,3.0,95.17,2
4,5,9,1,Delivered,"""business""",normal,email,6.0,131.74,2


## ML Dataset 2  Regression: Predicting Delivery Delay

This dataset supports a regression problem where the goal is to predict delivery delay 
(measured as arrival_date minus shipped_date). It integrates information from shipments, 
suppliers, warehouses, and shipment items.

Key features include:
- Supplier rating (from JSON)
- Warehouse location
- Number of products in the shipment
- Total units received
- Delivery delay in days (regression target)


In [7]:
regression_df = pd.read_sql_query("""
SELECT
    sh.shipment_id,
    sh.po_id,
    sh.warehouse_id,
    DATEDIFF(sh.arrival_date, sh.shipped_date) AS delivery_delay_days,
    s.name AS supplier_name,
    JSON_EXTRACT(s.supplier_metadata_json, '$.rating') AS supplier_rating,
    w.location AS warehouse_location,
    COUNT(si.product_id) AS num_products,
    SUM(si.quantity_received) AS total_units_received
FROM Shipments sh
JOIN Purchase_Orders po ON sh.po_id = po.po_id
JOIN Suppliers s ON po.supplier_id = s.supplier_id
JOIN Warehouses w ON sh.warehouse_id = w.warehouse_id
JOIN Shipment_Items si ON sh.shipment_id = si.shipment_id
WHERE sh.arrival_date IS NOT NULL
GROUP BY 
    sh.shipment_id, sh.po_id, sh.warehouse_id,
    delivery_delay_days, supplier_name, supplier_rating, warehouse_location;
""", engine)

regression_df.head()


,shipment_id,po_id,warehouse_id,delivery_delay_days,supplier_name,supplier_rating,warehouse_location,num_products,total_units_received
0,1,1,1,7,Maple Industrial Supplies Ltd.,4.6,"Niagara Falls, ON",2,693.0
1,16,16,3,8,Maple Industrial Supplies Ltd.,4.6,"Hamilton, ON",1,501.0
2,3,3,2,10,Northern Steel Components,4.4,"Toronto, ON",2,1000.0
3,18,18,4,9,Northern Steel Components,4.4,"Ottawa, ON",1,299.0
4,2,2,1,9,Great Lakes Packaging Corp.,4.7,"Niagara Falls, ON",1,298.0


## ML Dataset 3 Clustering: Supplier Performance Segmentation

This dataset is designed for clustering suppliers based on operational performance. 
It combines purchase order activity, units supplied, delivery delays, and supplier ratings.

Key clustering features include:
- Average supplier rating (from JSON)
- Total purchase orders
- Total units ordered
- Average delivery delay

This dataset enables segmentation of suppliers into performance groups.


In [8]:
clustering_df = pd.read_sql_query("""
SELECT
    s.supplier_id,
    s.name AS supplier_name,
    AVG(JSON_EXTRACT(s.supplier_metadata_json, '$.rating')) AS avg_rating,
    COUNT(po.po_id) AS total_purchase_orders,
    SUM(poi.quantity) AS total_units_ordered,
    AVG(DATEDIFF(sh.arrival_date, sh.shipped_date)) AS avg_delivery_delay
FROM Suppliers s
LEFT JOIN Purchase_Orders po ON s.supplier_id = po.supplier_id
LEFT JOIN Purchase_Order_Items poi ON po.po_id = poi.po_id
LEFT JOIN Shipments sh ON sh.po_id = po.po_id
GROUP BY 
    s.supplier_id, s.name;
""", engine)

clustering_df.head()


,supplier_id,supplier_name,avg_rating,total_purchase_orders,total_units_ordered,avg_delivery_delay
0,1,Maple Industrial Supplies Ltd.,4.6,3,1200.0,7.3333
1,2,Northern Steel Components,4.4,3,1300.0,9.6667
2,3,Great Lakes Packaging Corp.,4.7,2,700.0,9.0000
3,4,WestCoast Plastics & Resins,4.5,3,520.0,8.0000
4,5,Prairie Mechanical Parts,4.3,2,300.0,8.5000
